# SOXX 原始披露篮子：行尾脚注语法机械修复

只用原始名称，保留 Class A/B；不猜别名、CUSIP或真实进出，不读市场结果。

In [1]:
import json,hashlib,re,unicodedata,statistics
from pathlib import Path
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/stage1_soxx_reported_basket_rotation_footnote_repair_20261023';plan=json.loads((card/'plan.json').read_text());assert plan['frozen'] and not (card/'run_state.json').exists();sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
v1=root/plan['parent_v1']['card'];assert sha(v1/'plan.json')==plan['parent_v1']['plan_sha256'] and sha(v1/'result.json')==plan['parent_v1']['result_sha256'];old=json.loads((v1/'result.json').read_text());assert old['adjacent_transitions']==30
state={'plan_sha256':sha(card/'plan.json'),'phase':'source_read_attempted','original_source_result_load_attempts':6,'external_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
counts={'parenthesized_terminal_footnotes_removed':0,'bare_lowercase_terminal_footnotes_removed':0}
def normalize(name):
 s=unicodedata.normalize('NFKC',name).strip()
 while True:
  m=re.search(r'\s*\([a-z]\)\s*$',s)
  if not m:break
  s=s[:m.start()].rstrip();counts['parenthesized_terminal_footnotes_removed']+=1
 m=re.search(r'\s+[a-z](?:,[a-z])*\s*$',s)
 if m:s=s[:m.start()].rstrip();counts['bare_lowercase_terminal_footnotes_removed']+=1
 s=re.sub(r'[.,]',' ',s)
 return re.sub(r'\s+',' ',s).strip().casefold()
reports=[]
for source in plan['sources']:
 path=root/source['path'];assert sha(path)==source['sha256'];obj=json.loads(path.read_text())
 for report in obj['reports']:
  names=[normalize(x['security_original']) for x in report['equities']];assert len(names)==30 and len(set(names))==30,(report['accession'],names)
  reports.append({'period':report['report_date'],'accession':report['accession'],'names':set(names)})
assert len(reports)==31;reports.sort(key=lambda x:x['period']);v1_index={(x['prior_period'],x['next_period']):x for x in old['transition_details']};trans=[]
for a,b in zip(reports,reports[1:]):
 shared=a['names']&b['names'];prior_only=sorted(a['names']-b['names']);next_only=sorted(b['names']-a['names']);key=(a['period'],b['period']);assert key in v1_index
 trans.append({'prior_period':a['period'],'next_period':b['period'],'prior_accession':a['accession'],'next_accession':b['accession'],'v1_exact_overlap':v1_index[key]['exact_normalized_name_overlap'],'footnote_and_punctuation_overlap':len(shared),'prior_only_names':prior_only,'next_only_names':next_only})
values=[x['footnote_and_punctuation_overlap'] for x in trans];assert len(values)==30
result={'decision':'legacy_footnote_syntax_repair_improves_names_but_security_identity_still_unqualified','plan_sha256':sha(card/'plan.json'),'parent_v1_result_sha256':plan['parent_v1']['result_sha256'],'reports':len(reports),'adjacent_transitions':len(trans),'normalization_counts':counts,'v1_exact_overlap_min':old['exact_name_overlap_min'],'v1_exact_overlap_median':old['exact_name_overlap_median'],'repaired_overlap_min':min(values),'repaired_overlap_median':float(statistics.median(values)),'repaired_overlap_max':max(values),'repaired_transitions_below_25':sum(v<25 for v in values),'repaired_total_prior_only_name_slots':sum(len(x['prior_only_names']) for x in trans),'transitions_improved':sum(x['footnote_and_punctuation_overlap']>x['v1_exact_overlap'] for x in trans),'transitions_worsened':sum(x['footnote_and_punctuation_overlap']<x['v1_exact_overlap'] for x in trans),'within_report_collisions':0,'transition_details':trans,'CUSIP_or_legal_issuer_bridge_qualified':False,'weighted_turnover_feature_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Terminal footnotes and punctuation are source syntax, but repaired name equality is still not stable legal-security identity.','Remaining unmatched names require original CUSIP/issuer transition review before a turnover factor.','No 2019-2026 extension, predictive test or Stage1 account.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({k:result[k] for k in ('v1_exact_overlap_median','repaired_overlap_min','repaired_overlap_median','repaired_transitions_below_25','repaired_total_prior_only_name_slots','transitions_improved')})

{'v1_exact_overlap_median': 23.0, 'repaired_overlap_min': 19, 'repaired_overlap_median': 29.0, 'repaired_transitions_below_25': 1, 'repaired_total_prior_only_name_slots': 45, 'transitions_improved': 29}


# SOXX 披露篮子轮换：脚注语法修复后的身份边界

在固定的31份2011–2018原始 SEC 普通股报告上，另立只处理行尾 `(a)(b)` 和小写 `a,b/c` 脚注及非语义点号/逗号的机械来源版本。`Class A/B` 大写股类不剥离；每份30个名称均保持唯一。未请求新行情或原件、未读未来 SOXX 收益、未拟合或运行账户。

修复前30次相邻报告的严格名称交集最少9/30、中位数23/30；修复后最少 **19/30**、中位数 **29/30**、最多30/30。29次交集增加、0次变差，低于25的仅1次。行尾括号脚注移除96处、裸小写脚注移除322处；累计“前报告独有名称”槽位从277降为45。因此旧版的巨大表观轮换主要是原始脚注排版误差，不能用于策略。

唯一低交集发生在 **2013-07-31→2013-10-31**，两份报告有11个未匹配名称。其他相邻期也仍有45个未匹配槽位；它们可能是实际成分变化、公司更名、ADR/股类写法或合并。名称一致也不等于法律证券身份一致；披露市值变化还包含价格漂移，不可冒充基金换手。`result.json`逐期保留原始 accession 和修复后差异。

**决定：来源语法修复通过，证券身份门仍未通过。** 不计算加权换手、收益信息或 Stage1 0bp账户。下一步只针对未匹配项核原始 SEC 证券身份/股类与2013那次大变动，再与2019–2026 N-PORT 的 CUSIP体系接桥；在全期来源时点与身份闭合前，不把轮换作为新特征。
